# Phase 3 — Day 2: Novelty (Adaptive Residual Rescaling) on Kaggle T4

What this notebook does:

1. Clones the latest repo (must include the new `gate_alpha`/`gate_floor` flags in `evaluation.py` — push first if you haven't).
2. Runs a **grid search** over `gate_alpha ∈ {0.0, 0.1, 0.2, 0.3, 0.4, 0.5}` at 5-step DDIM on LOL-v2 Real (n=100).
3. Picks the best alpha by PSNR on LOL-v2 Real.
4. Re-evaluates with the best alpha on **both** eval15 and LOL-v2 Real, with LPIPS, to produce the ablation row that goes in Table 2 of the paper.
5. Prints the markdown ablation table.

Why this matters: this is the **novelty contribution** the area chair flagged as missing. Even if the best alpha turns out to be 0.0 (no improvement), reporting the negative result honestly is fine — what's not fine is having no ablation at all.

In [ ]:
# === Cell 1: install ===
!pip install -q --upgrade pip
!pip install -q scikit-image lpips matplotlib

In [ ]:
# === Cell 2: clone the repo (must contain the gate_alpha/gate_floor changes) ===
import os, sys, shutil, subprocess

BRANCH = "main"
REPO_URL = "https://github.com/chirana07/Diffusion_new_final.git"
REPO_DIR = "/kaggle/working/Diffunet"

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR],
               check=True)
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# Sanity: --gate-alpha must exist in the cloned evaluation.py
with open("evaluation.py") as f:
    has_gate = "--gate-alpha" in f.read()
print("gate_alpha flag present:", has_gate)
if not has_gate:
    raise SystemExit(
        "evaluation.py on GitHub does not have --gate-alpha. Push your local commits first, then re-run."
    )

In [ ]:
# === Cell 3: auto-discover dataset + checkpoint paths (same as Day 1) ===
import glob

def find_one(patterns, label):
    hits = sorted({h for pat in patterns for h in glob.glob(pat, recursive=True)})
    if not hits:
        print(f"  [{label}] NOT FOUND. Searched: {patterns}")
        return None
    print(f"  [{label}] = {hits[0]}")
    return hits[0]

EVAL15_LOW  = find_one(["/kaggle/input/**/eval15/low", "/kaggle/input/**/eval15/Low"],   "EVAL15_LOW")
EVAL15_HIGH = find_one(["/kaggle/input/**/eval15/high", "/kaggle/input/**/eval15/High"], "EVAL15_HIGH")
LOLV2_LOW   = find_one(["/kaggle/input/**/LOL-v2/Real_captured/Test/Low",
                         "/kaggle/input/**/Real_captured/Test/Low"], "LOLV2_LOW")
LOLV2_HIGH  = find_one(["/kaggle/input/**/LOL-v2/Real_captured/Test/Normal",
                         "/kaggle/input/**/Real_captured/Test/Normal"], "LOLV2_HIGH")

ckpt_hits = sorted(glob.glob("/kaggle/input/**/*.pth", recursive=True))
ckpt_hits = [c for c in ckpt_hits if any(k in c.lower() for k in ("final", "best", "last"))] or ckpt_hits
CHECKPOINT = ckpt_hits[0] if ckpt_hits else None
print(f"  [CHECKPOINT] = {CHECKPOINT}")

# === MANUAL OVERRIDE if any of these are wrong ===
# EVAL15_LOW  = "/kaggle/input/.../eval15/low"
# EVAL15_HIGH = "/kaggle/input/.../eval15/high"
# LOLV2_LOW   = "/kaggle/input/.../LOL-v2/Real_captured/Test/Low"
# LOLV2_HIGH  = "/kaggle/input/.../LOL-v2/Real_captured/Test/Normal"
# CHECKPOINT  = "/kaggle/input/.../final.pth"

missing = [n for n, v in [("EVAL15_LOW", EVAL15_LOW), ("EVAL15_HIGH", EVAL15_HIGH),
                            ("LOLV2_LOW", LOLV2_LOW), ("LOLV2_HIGH", LOLV2_HIGH),
                            ("CHECKPOINT", CHECKPOINT)] if not v]
if missing:
    raise SystemExit(f"Missing: {missing}. Attach the dataset or set the variable manually.")

EVAL_RESULTS = "/kaggle/working/eval_results_arr"
os.makedirs(EVAL_RESULTS, exist_ok=True)

# Splits used by both grid search and final ablation row
SPLITS_LOLV2 = [f"lolv2_real:{LOLV2_LOW}:{LOLV2_HIGH}"]
SPLITS_BOTH  = [
    f"eval15:{EVAL15_LOW}:{EVAL15_HIGH}",
    f"lolv2_real:{LOLV2_LOW}:{LOLV2_HIGH}",
]

In [ ]:
# === Cell 4: GPU sanity ===
import torch
print("cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - set Accelerator -> GPU T4")

In [ ]:
# === Cell 5: Grid search over gate_alpha at 5 DDIM steps on LOL-v2 Real ===
# We sweep alpha across {0.0, 0.1, ..., 0.5}. gate_floor stays at the default 0.5.
# alpha=0.0 reproduces vanilla DDIM exactly (sanity row).
import csv, sys, subprocess

ALPHAS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
GATE_FLOOR = 0.5
INFERENCE_STEPS = 5

grid_rows = []
for alpha in ALPHAS:
    tag = f"arr_a{int(alpha*100):03d}_s{INFERENCE_STEPS}"
    print(f"\n=== alpha={alpha} (tag={tag}) ===")
    cmd = [
        sys.executable, "evaluation.py",
        "--splits", *SPLITS_LOLV2,
        "--inference-steps", str(INFERENCE_STEPS),
        "--sampler", "ddim",
        "--checkpoint", CHECKPOINT,
        "--results-root", os.path.join(EVAL_RESULTS, "grid"),
        "--tag", tag,
        "--gate-alpha", str(alpha),
        "--gate-floor", str(GATE_FLOOR),
        "--no-lpips",  # skip LPIPS during the grid for speed; we'll add it for the final row
    ]
    print("$ " + " ".join(cmd))
    subprocess.run(cmd, check=True)

    summary_path = os.path.join(EVAL_RESULTS, f"grid_{tag}", "summary.csv")
    with open(summary_path) as f:
        for row in csv.DictReader(f):
            row["gate_alpha"] = alpha
            row["gate_floor"] = GATE_FLOOR
            grid_rows.append(row)

# Print grid table
print("\n### Grid search results (LOL-v2 Real, 5 DDIM steps)\n")
print("| alpha | floor | n | PSNR | SSIM | runtime/img (s) |")
print("|---|---|---|---|---|---|")
for r in grid_rows:
    print(f"| {float(r['gate_alpha']):.2f} | {r['gate_floor']} | {r['n']} | "
          f"{float(r['psnr_mean']):.3f} | {float(r['ssim_mean']):.4f} | "
          f"{float(r['runtime_mean']):.3f} |")

best = max(grid_rows, key=lambda r: float(r['psnr_mean']))
best_alpha = float(best['gate_alpha'])
print(f"\nBest alpha by PSNR: {best_alpha}  ({best['psnr_mean']} dB)")

In [ ]:
# === Cell 6: Final ablation row — best_alpha vs vanilla on BOTH splits, with LPIPS ===
ablation_rows = []
for alpha in (0.0, best_alpha):
    tag = f"final_a{int(alpha*100):03d}_s{INFERENCE_STEPS}"
    cmd = [
        sys.executable, "evaluation.py",
        "--splits", *SPLITS_BOTH,
        "--inference-steps", str(INFERENCE_STEPS),
        "--sampler", "ddim",
        "--checkpoint", CHECKPOINT,
        "--results-root", os.path.join(EVAL_RESULTS, "ablation"),
        "--tag", tag,
        "--gate-alpha", str(alpha),
        "--gate-floor", str(GATE_FLOOR),
    ]
    print("$ " + " ".join(cmd))
    subprocess.run(cmd, check=True)
    summary_path = os.path.join(EVAL_RESULTS, f"ablation_{tag}", "summary.csv")
    with open(summary_path) as f:
        for row in csv.DictReader(f):
            row["variant"] = "vanilla DDIM" if alpha == 0.0 else f"+ ARR (\u03B1={alpha})"
            ablation_rows.append(row)

In [ ]:
# === Cell 7: Print the paper's ablation table ===
print("\n### Ablation: Adaptive Residual Rescaling at 5 DDIM steps\n")
print("| Variant | Split | n | PSNR | SSIM | LPIPS | Latency/img (s) |")
print("|---|---|---|---|---|---|---|")
for r in ablation_rows:
    lp = r.get('lpips_mean')
    lp_s = f"{float(lp):.4f}" if lp not in (None, '', 'None') else '-'
    print(f"| {r['variant']} | {r['split']} | {r['n']} | "
          f"{float(r['psnr_mean']):.3f} | {float(r['ssim_mean']):.4f} | "
          f"{lp_s} | {float(r['runtime_mean']):.3f} |")

In [ ]:
# === Cell 8: Bundle outputs ===
import shutil, json

# Save a small JSON with the picked alpha so we don't lose it
with open(os.path.join(EVAL_RESULTS, "best_alpha.json"), "w") as f:
    json.dump({"best_alpha": best_alpha,
                "gate_floor": GATE_FLOOR,
                "inference_steps": INFERENCE_STEPS,
                "grid": [{k: r[k] for k in ("gate_alpha", "split", "psnr_mean", "ssim_mean", "runtime_mean")}
                          for r in grid_rows]}, f, indent=2)

out_zip = "/kaggle/working/phase3_day2_outputs.zip"
if os.path.exists(out_zip):
    os.remove(out_zip)
shutil.make_archive(out_zip.replace('.zip', ''), 'zip', EVAL_RESULTS)
print("Wrote", out_zip)
print("Download from Kaggle right panel -> Output tab.")